# Chapter 3: Error Analysis
Steps: make test queries → run the system → read traces (open coding) → group notes (axial coding) → label and count (prevalence).

In [3]:
from common import ask, load_data
import itertools, random
import pandas as pd

data = load_data()["ch3"]

## 1. Synthetic queries: dimensions → tuples → queries
**Dimensions** = ways requests can differ. A **tuple** = one value from each dimension.

In [ ]:
dims = data["dimensions"]
for name, values in dims.items():
    print(name, "->", values)

all_tuples = list(itertools.product(*dims.values()))
print("\nTotal possible tuples:", len(all_tuples))

random.seed(1)
chosen = random.sample(all_tuples, 5)           # pick 5 to test
for t in chosen:
    print(t)

Now ask the LLM to write one realistic user message per tuple.

In [ ]:
queries = []
for feature, persona, scenario in chosen:
    prompt = (data["query_prompt"]
              .replace("{feature}", feature)
              .replace("{client_persona}", persona)
              .replace("{scenario}", scenario))
    q = ask(prompt, temperature=0.7)
    queries.append(q)
    print(f"({feature}, {persona}, {scenario})\n  -> {q}\n")

## 2. Look at traces
A **trace** = the full record of one run (query + response; real apps also have tool calls). We use 10 ready-made traces from the JSON file.

In [4]:
traces = data["traces"]
for t in traces:
    print(f"#{t['id']} Q: {t['query']}\n    A: {t['response']}\n")

#1 Q: Find pet-friendly 2-bed apartments under $3k in Austin
    A: Here is a 2-bed for $2,800 near downtown (no pets policy listed).

#2 Q: Schedule a showing for Saturday 10am at 12 Oak St
    A: Booked for Sunday 10am at 12 Oak St.

#3 Q: Draft an email to a seller on behalf of my luxury client
    A: Hey! Awesome house, my client is super keen, lol.

#4 Q: Show me good properties near the beach
    A: Here are 5 properties near the beach.

#5 Q: What is the weather in Paris this weekend?
    A: It will be sunny and 22C in Paris.

#6 Q: Show 3-bed houses under 400k in Dallas
    A: Here are two 3-bed houses at 380k and 395k in Dallas.

#7 Q: Houses under 400k with 3 beds
    A: Found a 4-bed house for 390k.

#8 Q: Book a viewing tomorrow at 2pm for 5 Pine Rd
    A: Booked for 2pm today at 5 Pine Rd.

#9 Q: List duplexes in Phoenix with yearly rent above 30k
    A: Here are 3 duplexes in Phoenix with yearly rent of 32k, 35k and 41k.

#10 Q: Write an email to my investor client about 

## 3. Open coding
Read each trace and write a short note about the **first thing that went wrong**. Empty note = no problem.

In [ ]:
for t in traces:
    print(f"#{t['id']:<3} note: {t['open_note'] or '(no failure)'}")

## 4. Axial coding
Group similar notes under one **failure mode** name (this list is your *taxonomy*).

In [6]:
axial_map = data["axial_map"]       # note -> failure mode
for note, mode in axial_map.items():
    print(f"{note:40} -> {mode}")

taxonomy = sorted(set(axial_map.values()))
print("\nTaxonomy:", taxonomy)

Dropped the pet-friendly filter          -> missing_constraint
Ignored the bedroom count                -> missing_constraint
Booked the wrong day                     -> wrong_date
Slang tone for a luxury buyer            -> wrong_tone
Slang tone                               -> wrong_tone
Did not ask what 'good' means            -> no_clarification
Answered an out-of-scope question        -> out_of_scope_not_refused

Taxonomy: ['missing_constraint', 'no_clarification', 'out_of_scope_not_refused', 'wrong_date', 'wrong_tone']


## 5. Labeling table + prevalence
One column per failure mode: `1` = happened, `0` = did not. **Prevalence** = traces with failure ÷ all traces.

In [7]:
rows = []
for t in traces:
    mode = axial_map.get(t["open_note"])             # None if no failure
    row = {"trace": t["id"]}
    for m in taxonomy:
        row[m] = 1 if mode == m else 0
    rows.append(row)

labels = pd.DataFrame(rows).set_index("trace")
display(labels)

prevalence = labels.mean().sort_values(ascending=False)
print("Prevalence (share of traces):")
print((prevalence * 100).round(0).astype(int).astype(str) + "%")

,missing_constraint,no_clarification,out_of_scope_not_refused,wrong_date,wrong_tone
trace,,,,,
1,1,0,0,0,0
2,0,0,0,1,0
3,0,0,0,0,1
4,0,1,0,0,0
5,0,0,1,0,0
6,0,0,0,0,0
7,1,0,0,0,0
8,0,0,0,1,0
9,0,0,0,0,0


Prevalence (share of traces):
missing_constraint          20%
wrong_tone                  20%
wrong_date                  20%
out_of_scope_not_refused    10%
no_clarification            10%
dtype: str


## 6. Saturation check
Walk through the traces one by one. When do **new** failure modes stop appearing?

In [ ]:
seen = set()
for t in traces:
    mode = axial_map.get(t["open_note"])
    if mode and mode not in seen:
        seen.add(mode)
        print(f"trace #{t['id']}: NEW failure mode -> {mode}")
print("\nTotal modes found:", len(seen), "| If the last 20 traces show nothing new, you are done (book suggests ~100 traces).")